In [ ]:
pip install pandas beautifulsoup4 lxml selenium webdriver-manager


In [ ]:

pip install -U bottleneck


In [ ]:
pip install --upgrade pip
pip install selenium webdriver-manager pandas beautifulsoup4 lxml xlsxwriter
python macrotrends_quarterly_all.py


In [ ]:
pip install selenium-wire

In [ ]:
pip install blinker


In [ ]:
pip install --upgrade pip


In [ ]:
pip install selenium webdriver-manager pandas beautifulsoup4 lxml xlsxwriter


In [ ]:
pip install selenium webdriver-manager pandas beautifulsoup4 lxml xlsxwriter


In [ ]:
pip install -U selenium webdriver-manager


In [ ]:
# 8/22 코드

In [ ]:
pip install selenium webdriver-manager pandas beautifulsoup4 openpyxl


#  이코드 완벽 baseline

In [ ]:
# import time, re, json
# import pandas as pd
# from selenium import webdriver
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.chrome.options import Options
# from bs4 import BeautifulSoup
# from webdriver_manager.chrome import ChromeDriverManager

# # 설정
# TICKER = "TSLA"
# FREQ = "Q"  # Q: 분기 / A: 연간
# URL = f"https://www.macrotrends.net/stocks/charts/{TICKER}/tesla/key-financial-ratios?freq={FREQ}"
# CSV_PATH = f"{TICKER}_financial_ratios_{FREQ}.csv"

# # 브라우저 세팅
# opts = Options()
# opts.add_argument("--window-size=1920,1200")
# opts.add_argument("--disable-gpu")
# # opts.add_argument("--headless")  # 필요하면 활성화

# driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
# driver.get(URL)
# time.sleep(5)

# # HTML 파싱
# soup = BeautifulSoup(driver.page_source, "html.parser")
# driver.quit()

# # originalData 추출
# script_tags = soup.find_all("script", string=re.compile(r"originalData"))
# target = None
# for sc in script_tags:
#     match = re.search(r"var originalData = (\[.*?\]);", sc.string, re.S)
#     if match:
#         target = match.group(1)
#         break
# if not target:
#     raise ValueError("originalData not found")

# data = json.loads(target)

# # 데이터프레임 생성
# rows = []
# for entry in data:
#     row = {"Metric": BeautifulSoup(entry.get("field_name", ""), "html.parser").get_text()}
#     for k, v in entry.items():
#         if re.match(r"\d{4}-\d{2}-\d{2}", k):
#             row[k] = v
#     rows.append(row)

# df = pd.DataFrame(rows).set_index("Metric")
# df = df.sort_index(axis=1, ascending=False)  # 최신 날짜가 왼쪽에 오도록 정렬

# # CSV 저장
# df.to_csv(CSV_PATH, encoding="utf-8-sig")
# print(f"✅ Saved CSV: {CSV_PATH}")


In [ ]:
# Revision

In [ ]:
import time
import re
import json
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager

# 🔹 티커와 URL용 회사명 매핑
ticker_names = {
    "GOOG": "alphabet", 
    "BN": "brookfield", 
    "OXY": "occidental-petroleum",
    "CRSP": "crispr-therapeutics-ag",
    "NVO": "novo-nordisk",
    "PATH": "uipath",
    "OKTA": "okta", 
    "ILMN": "illumina",
    "TSM": "taiwan-semiconductor-manufacturing", 
    "CRCL": "circle-internet",
    "NTRA": "natera",
    "META": "meta-platforms", 
    "NU": "nu-holdings",
     "CPNG": "coupang", 
    "CVX": "chevron",
    "PLTR": "palantir-technologies", 
    "DDOG": "datadog", 
    "CRWD": "crowdstrike",
    "AVGO": "broadcom",
    "UNH": "unitedhealth-group", 
    "LMND": "lemonade",
    "AAPL": "apple", 
    "MITK": "mitek-systems",
    "MSFT": "microsoft", 
    "BBAI": "bigbearai-holdings",
    "VST": "vistra", 
    "VRT": "vertiv-holdings",
    "MP": "mp-materials",
    "COST": "costco",
    "TEVA": "teva-pharmaceutical-industries",
    "TSLA": "tesla",
    "U": "unity-software",
    "RDDT": "reddit",
    "SNOW": "snowflake",
    "RKLB": "rocket-lab",
    "MELI": "mercadolibre",
    "EH": "ehang-holdings",
    "GRAL": "grail",
    "PLUG": "plug-power",
    "SE": "sea",
    "TEM": "tempus-ai",
    "IREN": "iren",
    "PI": "impinj",
    "JOBY": "joby-aviation",
    "APP":"applovin",
    "COIN": "coinbase-gloabl",
    "NVDA": "nvidia"
}

ticker_names = {
   "GRAL": "Grail",
    "COIN": "coinbase-gloabl",
}
freq = "Q"
save_folder = r"C:\Users\seung\OneDrive\주식\Financial_Data_real"
os.makedirs(save_folder, exist_ok=True)

# Selenium 설정
opts = Options()
opts.add_argument("--window-size=1920,1200")
# opts.add_argument("--headless")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

statements = {
    "Income Statement": "income-statement",
    "Balance Sheet": "balance-sheet",
    "Cash Flow Statement": "cash-flow-statement",
    "Key Financial Ratios": "financial-ratios"
}

def fetch_statement_df(ticker, company_slug, statement, freq="Q"):
    url = f"https://www.macrotrends.net/stocks/charts/{ticker}/{company_slug}/{statement}?freq={freq}"
    driver.get(url)
    time.sleep(5)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    script_tags = soup.find_all("script", string=re.compile(r"originalData"))
    target = None
    for sc in script_tags:
        m = re.search(r"var originalData = (\[.*?\]);", sc.string, re.S)
        if m:
            target = m.group(1)
            break
    if not target:
        raise ValueError(f"originalData not found for {ticker} - {statement}")
    data = json.loads(target)
    rows = []
    for e in data:
        row = {"Metric": BeautifulSoup(e.get("field_name", ""), "html.parser").get_text()}
        for k, v in e.items():
            if re.match(r"\d{4}-\d{2}-\d{2}", k):
                row[k] = v
        rows.append(row)
    df = pd.DataFrame(rows).set_index("Metric")
    return df.sort_index(axis=1, ascending=False)

# ▶ 모든 티커 순회
for ticker, name_slug in ticker_names.items():
    print(f"\n🔄 Processing {ticker} ({name_slug})...")

    dfs = {}
    for sheet_name, path in statements.items():
        print(f"   📊 Fetching: {sheet_name}...")
        try:
            df = fetch_statement_df(ticker, name_slug, path, freq)
            dfs[sheet_name] = df
        except Exception as e:
            print(f"   ❌ Error fetching {sheet_name} for {ticker}: {e}")

    # 저장
    if dfs:
        output_file = os.path.join(save_folder, f"{ticker}_financials_{freq}.xlsx")
        with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
            for sheet, df in dfs.items():
                df.to_excel(writer, sheet_name=sheet)
        print(f"   ✅ 저장 완료: {output_file}")
    else:
        print(f"   ⚠️ No data for {ticker}")

driver.quit()


In [ ]:
# # Data Analysis Code
# 필수 아님(있으면 정확도↑)

# price : 현재 주가(USD) → 시가총액, 멀티플, EV 계산에 사용

# beta, rf, mrp, pretax_cost_of_debt : WACC 계산용(없으면 합리적 디폴트)

# rev_g_list : 5년 매출 성장률 경로(예: [0.08,0.07,0.06,0.05,0.04]) — 넣지 않으면 최근 TTM YoY로 고정

# ebit_margin, da_pct, capex_pct, dnwc_pct : 각각 Revenue 대비 비율. 안 넣으면 최근 TTM 관측치로 자동 세팅

# wacc_override : 직접 WACC 지정하고 싶을 때

# ltg : 터미널 성장률(고든)

# exit_multiple : 대신 EV/EBITDA 멀티플로 종료하고 싶을 때(선택)

# CCC 완전 계산을 위해 아래 중 하나가 파일에 있어야 함(있으면 자동 사용)

# Balance Sheet에 Accounts Payable(또는 ‘Payables/Trade Payables’ 유사명)

# OR Ratios에 Payables Turnover(혹은 Days Payable Outstanding)
# 없으면 DPO는 계산하지 않고 CCC는 NaN으로 둔다.

In [ ]:

########################
# -*- coding: utf-8 -*-
"""
{TICKER}_financials_Q.xlsx -> {TICKER}_analysis_Q.xlsx
- 시트명 자동 인식(Income Statement / Balance Sheet / Cash Flow Statement / Key Financial Ratios 등 변형 대응)
- CCC(+가능하면 DPO), 멀티플, 비율기반 FCFF 예측/DCF 포함
"""

import os, re, math
import numpy as np
import pandas as pd

# ========================= 사용자 설정 =========================
TICKER   = "AAPL"
BASE_DIR = r"C:\Users\seung\OneDrive\주식\Financial_Data_real"
INPUT_XLSX  = os.path.join(BASE_DIR, f"{TICKER}_financials_Q.xlsx")
OUTPUT_XLSX = os.path.join(BASE_DIR, f"{TICKER}_analysis_Q.xlsx")

CONFIG = {
    # ---- Valuation inputs ----
    "price": None,          # ex) 230.5  (없으면 멀티플/EV 일부 제한)
    "beta": 1.2,
    "rf": 0.045,            # 10Y 등
    "mrp": 0.055,
    "pretax_cost_of_debt": 0.05,

    # ---- DCF/모델 가정 ----
    # 5년 매출 성장률 경로. None이면 최근 TTM YoY로 고정
    "rev_g_list": None,     # 예: [0.08,0.07,0.06,0.05,0.04]
    # 아래 4개 비율(Revenue 기준). None이면 최근 TTM 관측치로 자동
    "ebit_margin": None,    # EBIT_TTM / Revenue_TTM
    "da_pct": None,         # D&A_TTM / Revenue_TTM
    "capex_pct": None,      # CapEx_TTM / Revenue_TTM
    "dnwc_pct": None,       # ΔNWC_TTM / Revenue_TTM  (없으면 0으로 둠)

    "tax_rate_override": None,  # 없으면 최근 분기 유효세율 사용
    "wacc_override": None,      # 직접 지정하고 싶으면 입력
    "ltg": 0.025,               # 터미널 성장률 (고든)
    "exit_multiple": None       # EV/EBITDA Exit multiple (고든 대신 사용)
}
# ===============================================================


# ---------------- 시트 자동 탐색 ----------------
def _norm(s: str) -> str:
    """소문자화 + 영숫자만 남기기"""
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

def pick_sheet(xlsx_path: str, wanted_key: str) -> str:
    """
    파일 내 시트 이름을 스캔해서 우리가 원하는 분류의 시트를 찾아 실제 시트명 반환.
    wanted_key: 'income_statement' | 'balance_sheet' | 'cash_flow' | 'key_financial_ratios'
    """
    aliases = {
        "income_statement":      ["incomestatement", "income", "incomestmt"],
        "balance_sheet":         ["balancesheet", "balance"],
        "cash_flow":             ["cashflowstatement", "cashflow", "cashflowstmt"],
        "key_financial_ratios":  ["keyfinancialratios", "financialratios", "ratios"],
    }
    xl = pd.ExcelFile(xlsx_path)
    norm_map = {name: _norm(name) for name in xl.sheet_names}
    for real_name, normed in norm_map.items():
        for cand in aliases[wanted_key]:
            if cand in normed:
                return real_name
    raise ValueError(f"[{wanted_key}] 시트를 찾지 못했습니다. 시트 목록: {xl.sheet_names}")


# ---------- 유틸 ----------
def to_num(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return np.nan
    s = str(x).strip()
    if s in ("", "-", "—"): return np.nan
    neg = False
    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1]; neg = True
    s = s.replace(",", "").replace("$", "")
    try:
        v = float(s);  return -v if neg else v
    except:
        return np.nan

def norm_cols(df):
    cols = []
    for c in df.columns:
        try:
            d = pd.to_datetime(c, errors="raise")
            cols.append((c, d))
        except:
            pass
    if not cols: return df.iloc[:, 0:0]
    cols.sort(key=lambda t: t[1])  # 과거 -> 최근
    return df[[c for c,_ in cols]]

def read_sheet(xlsx, wanted_key):
    """원하는 분류 키로 시트 찾아서 읽기 + 숫자화 + 날짜열만 정렬"""
    sheet_name = pick_sheet(xlsx, wanted_key)
    df = pd.read_excel(xlsx, sheet_name=sheet_name)
    if "Metric" not in df.columns:
        df.rename(columns={df.columns[0]:"Metric"}, inplace=True)
    df["Metric"] = df["Metric"].astype(str).str.strip()
    df.set_index("Metric", inplace=True)
    for c in df.columns: df[c] = df[c].map(to_num)
    return norm_cols(df)

def get_row(df, keys, default=np.nan):
    if df is None or df.empty:
        return pd.Series([default]*0)
    idx_map = {i.lower().strip(): i for i in df.index}
    low_index = list(idx_map.keys())
    # 완전일치
    for k in keys:
        lk = k.lower().strip()
        if lk in idx_map: return df.loc[idx_map[lk]]
    # 부분매칭
    for k in keys:
        lk = k.lower().strip()
        for li in low_index:
            if lk in li:
                return df.loc[idx_map[li]]
    return pd.Series([default]*df.shape[1], index=df.columns, name=keys[0])

def ttm(series):
    s = series.dropna()
    return s.iloc[-4:].sum() if len(s)>=4 else np.nan

def yoy_growth_ttm(series):
    s = series.dropna()
    if len(s) < 8: return np.nan
    cur = s.iloc[-4:].sum()
    prv = s.iloc[-8:-4].sum()
    return cur/prv - 1 if prv not in (0, np.nan) else np.nan

def qoq(series):
    s = series.dropna()
    if len(s) < 2: return np.nan
    prev, cur = s.iloc[-2], s.iloc[-1]
    return (cur/prev - 1) if prev not in (0, np.nan) else np.nan

# WACC
def compute_wacc(price, shares, debt, cash, beta, rf, mrp, cod, tax_rate):
    coe = rf + beta*mrp if (beta is not None and rf is not None and mrp is not None) else None
    if price and shares:
        mc = price*shares
    else:
        mc = None
    if mc is None or debt is None or (mc+debt)==0 or coe is None:
        return coe, cod, None, mc
    V = mc + debt
    w_e, w_d = mc/V, debt/V
    wacc = w_e*coe + w_d*cod*(1 - (tax_rate if tax_rate is not None else 0.21))
    return coe, cod, wacc, mc


# ---------- 입력 파일 로드 (시트 자동 인식) ----------
inc = read_sheet(INPUT_XLSX, "income_statement")
bal = read_sheet(INPUT_XLSX, "balance_sheet")
cfs = read_sheet(INPUT_XLSX, "cash_flow")
rat = read_sheet(INPUT_XLSX, "key_financial_ratios")

dates = inc.columns
latest = dates[-1] if len(dates)>0 else None

# ---------- 핵심 시리즈 ----------
Revenue   = get_row(inc, ["Revenue"])
COGS      = get_row(inc, ["Cost Of Goods Sold"])
Gross     = get_row(inc, ["Gross Profit"])
EBIT      = get_row(inc, ["EBIT"])
EBITDA    = get_row(inc, ["EBITDA"])
NetIncome = get_row(inc, ["Net Income"])
EPS       = get_row(inc, ["Basic EPS","EPS - Earnings Per Share"])
Shares    = get_row(inc, ["Shares Outstanding","Basic Shares Outstanding"])

OCF       = get_row(cfs, ["Cash Flow From Operating Activities"])
DA        = get_row(cfs, ["Total Depreciation And Amortization"])
CapEx_raw = get_row(cfs, ["Net Change In Property, Plant, And Equipment"])
CapEx     = -CapEx_raw.clip(upper=0)  # 유출(+)

Cash      = get_row(bal, ["Cash On Hand","Cash"])
CurrAssets= get_row(bal, ["Total Current Assets"])
CurrLiab  = get_row(bal, ["Total Current Liabilities"])
LT_Debt   = get_row(bal, ["Long Term Debt","Long-Term Debt"])
TotAssets = get_row(bal, ["Total Assets"])
TotLiab   = get_row(bal, ["Total Liabilities"])
Equity    = get_row(bal, ["Share Holder Equity","Total Shareholder Equity"])

# Turnover들
InvTurn       = get_row(rat, ["Inventory Turnover Ratio"])
RecvTurn      = get_row(rat, ["Receiveable Turnover","Receivable Turnover"])
PayTurn_ratio = get_row(rat, ["Payables Turnover","Accounts Payables Turnover"])
DSO_rat       = get_row(rat, ["Days Sales In Receivables"])

# Balance에서 Payables 찾기(있을 때)
Payables = get_row(bal, ["Accounts Payable","Trade Payables","Payables"])

# ---------- TTM/성장 ----------
Revenue_TTM   = ttm(Revenue)
EBITDA_TTM    = ttm(EBITDA)
EBIT_TTM      = ttm(EBIT)
NetIncome_TTM = ttm(NetIncome)
EPS_TTM       = ttm(EPS)
OCF_TTM       = ttm(OCF)
DA_TTM        = ttm(DA)
CapEx_TTM     = ttm(CapEx)

RevYoY_TTM    = yoy_growth_ttm(Revenue)
EPS_QoQ       = qoq(EPS)
Rev_QoQ       = qoq(Revenue)

# 유효세율
Tax          = get_row(inc, ["Income Taxes"])
PreTax       = get_row(inc, ["Pre-Tax Income"])
tax_rate_eff = CONFIG["tax_rate_override"]
if tax_rate_eff is None:
    tax_rate_eff = (Tax.iloc[-1]/PreTax.iloc[-1]) if (not pd.isna(Tax.iloc[-1]) and not pd.isna(PreTax.iloc[-1]) and PreTax.iloc[-1]!=0) else 0.21

# NWC (근사: (CA − Cash) − CL)
NWC = (CurrAssets - Cash) - CurrLiab
dNWC_latest = np.nan
if len(NWC.dropna())>=2:
    dNWC_latest = NWC.iloc[-1]-NWC.iloc[-2]

# FCFF series & TTM
def fcff_series(EBIT_s, DA_s, CapEx_s, NWC_s, tax_rate):
    out=[]
    for i in range(len(EBIT_s)):
        e = EBIT_s.iloc[i]
        da= DA_s.iloc[i] if i<len(DA_s) else np.nan
        cx= CapEx_s.iloc[i] if i<len(CapEx_s) else np.nan
        dNWC=np.nan
        if i>=1 and i < len(NWC_s):
            dNWC = NWC_s.iloc[i]-NWC_s.iloc[i-1]
        if any(pd.isna([e,da,cx])):
            out.append(np.nan); continue
        out.append(e*(1-tax_rate)+da-cx-(0 if pd.isna(dNWC) else dNWC))
    return pd.Series(out, index=EBIT_s.index)

FCFF_s   = fcff_series(EBIT, DA, CapEx, NWC, tax_rate_eff)
FCFF_TTM = ttm(FCFF_s)
FCFF_q   = FCFF_s.iloc[-1]

# ---------- CCC (DIO, DSO, DPO) ----------
DIO = (365/InvTurn.iloc[-1]) if not (pd.isna(InvTurn.iloc[-1]) or InvTurn.iloc[-1]==0) else np.nan
DSO = DSO_rat.iloc[-1] if not pd.isna(DSO_rat.iloc[-1]) else (365/RecvTurn.iloc[-1] if not pd.isna(RecvTurn.iloc[-1]) and RecvTurn.iloc[-1]!=0 else np.nan)

# DPO: 1) ratios에 Payables Turnover가 있으면 365/PayTurn  2) 없으면 Payables/COGS로 근사
DPO = np.nan
if not pd.isna(PayTurn_ratio.iloc[-1]) and PayTurn_ratio.iloc[-1]!=0:
    DPO = 365/PayTurn_ratio.iloc[-1]
else:
    ap_series = Payables.dropna()
    COGS_TTM = ttm(COGS)
    if len(ap_series)>=1 and not pd.isna(COGS_TTM) and COGS_TTM!=0:
        avg_ap = ap_series.iloc[-4:].mean() if len(ap_series)>=4 else ap_series.iloc[-1]
        DPO = (avg_ap/COGS_TTM)*365

CCC = (DIO + DSO - DPO) if not any(pd.isna([DIO,DSO,DPO])) else np.nan

# ---------- 멀티플 ----------
price  = CONFIG["price"]
shares = Shares.iloc[-1] if not pd.isna(Shares.iloc[-1]) else None
mktcap = price*shares if (price and shares) else None
cash_v = Cash.iloc[-1] if not pd.isna(Cash.iloc[-1]) else None
debt_v = LT_Debt.iloc[-1] if not pd.isna(LT_Debt.iloc[-1]) else None
EV     = (mktcap + debt_v - cash_v) if (mktcap and debt_v is not None and cash_v is not None) else None

P_E   = (price/(EPS_TTM if EPS_TTM else np.nan)) if price and EPS_TTM else np.nan
P_S   = (mktcap/Revenue_TTM) if (mktcap and Revenue_TTM) else np.nan
P_B   = np.nan
BVPS  = get_row(rat, ["Book Value Per Share"]).iloc[-1]
if price and not pd.isna(BVPS) and BVPS!=0: P_B = price/BVPS

P_FCF = np.nan
if price and shares and FCFF_TTM and FCFF_TTM!=0:
    P_FCF = price / (FCFF_TTM/shares)

EV_EBITDA = (EV/EBITDA_TTM) if (EV and EBITDA_TTM and EBITDA_TTM!=0) else np.nan
EV_EBIT   = (EV/EBIT_TTM)   if (EV and EBIT_TTM and EBIT_TTM!=0)     else np.nan

# ---------- WACC ----------
beta = CONFIG["beta"]; rf=CONFIG["rf"]; mrp=CONFIG["mrp"]; cod=CONFIG["pretax_cost_of_debt"]
coE, coD, wacc_calc, mc_tmp = compute_wacc(price, shares, debt_v, cash_v, beta, rf, mrp, cod, tax_rate_eff)
WACC = CONFIG["wacc_override"] if CONFIG["wacc_override"] is not None else wacc_calc

# ---------- 비율 기반 FCFF 예측 모델 ----------
ebit_margin = CONFIG["ebit_margin"] if CONFIG["ebit_margin"] is not None else (EBIT_TTM/Revenue_TTM if Revenue_TTM else 0.2)
da_pct      = CONFIG["da_pct"]      if CONFIG["da_pct"]      is not None else (DA_TTM/Revenue_TTM   if Revenue_TTM and DA_TTM else 0.04)
capex_pct   = CONFIG["capex_pct"]   if CONFIG["capex_pct"]   is not None else (CapEx_TTM/Revenue_TTM if Revenue_TTM and CapEx_TTM else 0.04)
dnwc_pct    = CONFIG["dnwc_pct"]    if CONFIG["dnwc_pct"]    is not None else 0.00
rev_g_list  = CONFIG["rev_g_list"]  if CONFIG["rev_g_list"]  is not None else [RevYoY_TTM or 0.05]*5

rev0 = Revenue_TTM
proj = []; fcff_proj = []; EBIT_proj = []
for t,g in enumerate(rev_g_list, start=1):
    rev_t = (rev0 if t==1 else proj[-1])*(1+g)
    ebit_t  = rev_t * ebit_margin
    nopat_t = ebit_t * (1 - (tax_rate_eff if tax_rate_eff is not None else 0.21))
    da_t    = rev_t * da_pct
    capex_t = rev_t * capex_pct
    dnwc_t  = rev_t * dnwc_pct
    fcff_t  = nopat_t + da_t - capex_t - dnwc_t
    proj.append(rev_t); EBIT_proj.append(ebit_t); fcff_proj.append(fcff_t)

# Terminal Value
if CONFIG["exit_multiple"] and EBITDA_TTM:
    tv = (EBITDA_TTM * CONFIG["exit_multiple"])
else:
    tv = (fcff_proj[-1]*(1+CONFIG["ltg"])) / (WACC - CONFIG["ltg"]) if (WACC and CONFIG["ltg"] is not None and WACC>CONFIG["ltg"]) else np.nan

# 할인
DCF_tbl = pd.DataFrame({"Year":[f"Y{i}" for i in range(1, len(rev_g_list)+1)],
                        "Revenue":proj,"EBIT":[*EBIT_proj],"FCFF":[*fcff_proj]})
if WACC:
    DCF_tbl["DF"] = [(1+WACC)**i for i in range(1,len(rev_g_list)+1)]
    DCF_tbl["PV"] = DCF_tbl["FCFF"]/DCF_tbl["DF"]
    PV_TV = tv/((1+WACC)**len(rev_g_list)) if not pd.isna(tv) else np.nan
    EV_model = DCF_tbl["PV"].sum() + (PV_TV if not pd.isna(PV_TV) else 0)
else:
    EV_model = np.nan

# Equity Value & Implied Price
ImpliedPrice = np.nan
if not pd.isna(EV_model):
    equity_val = EV_model + (cash_v if cash_v is not None else 0) - (debt_v if debt_v is not None else 0)
    if shares and shares!=0: ImpliedPrice = equity_val/shares

# ---------- 시트 구성 ----------
snapshot = pd.DataFrame({
    "Metric":[
        "Latest Date","Revenue latest","Revenue TTM","Revenue YoY (TTM)","Rev QoQ",
        "EPS latest","EPS TTM","EPS QoQ",
        "EBITDA TTM","EBIT TTM","NetIncome TTM",
        "FCFF latest q","FCFF TTM",
        "DIO","DSO","DPO","CCC",
        "Tax Rate","WACC (calc/override)"],
    "Value":[
        latest, Revenue.iloc[-1], Revenue_TTM, RevYoY_TTM, Rev_QoQ,
        EPS.iloc[-1], EPS_TTM, EPS_QoQ,
        EBITDA_TTM, EBIT_TTM, NetIncome_TTM,
        FCFF_q, FCFF_TTM,
        DIO, DSO, DPO, CCC,
        tax_rate_eff, WACC
    ]
})

multiples = pd.DataFrame({
    "Multiple":["Price","Shares","Market Cap","EV",
                "P/E (TTM)","P/S (TTM)","P/B","P/FCF (TTM)","EV/EBITDA (TTM)","EV/EBIT (TTM)"],
    "Value":[price, shares, mktcap, EV, P_E, P_S, P_B, P_FCF, EV_EBITDA, EV_EBIT]
})

ccc_sheet = pd.DataFrame({
    "Item":["Inv Turnover","Recv Turnover","Payables Turnover",
            "DIO(365/InvTurn)","DSO","DPO","CCC=DIO+DSO-DPO"],
    "Value":[InvTurn.iloc[-1], RecvTurn.iloc[-1], PayTurn_ratio.iloc[-1],
             DIO, DSO, DPO, CCC]
})

assumptions = pd.DataFrame({
    "Assumption":["Revenue0 (TTM)","rev_g_list (Y1~Y5)","EBIT margin","DA % of Rev","CapEx % of Rev","ΔNWC % of Rev",
                  "Tax rate","WACC (calc)","WACC override","Terminal g","Exit multiple","beta","rf","mrp","pre-tax CoD","price"],
    "Value":[Revenue_TTM, str(rev_g_list), ebit_margin, da_pct, capex_pct, dnwc_pct,
             tax_rate_eff, WACC, CONFIG["wacc_override"], CONFIG["ltg"], CONFIG["exit_multiple"],
             CONFIG["beta"], CONFIG["rf"], CONFIG["mrp"], CONFIG["pretax_cost_of_debt"], price]
})

valuation = pd.DataFrame({
    "Metric":["EV (Model)","Terminal Value","Equity Value (Model)","Implied Price (Model)"],
    "Value":[EV_model, tv,
             (EV_model + (cash_v if cash_v else 0) - (debt_v if debt_v else 0)) if not pd.isna(EV_model) else np.nan,
             ImpliedPrice]
})

with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as w:
    snapshot.to_excel(w, "SnapShot", index=False)
    multiples.to_excel(w, "Multiples", index=False)
    ccc_sheet.to_excel(w, "CCC", index=False)

    # TTM & Growth
    ttm_growth = pd.DataFrame({
        "Item":["Revenue","NetIncome","OCF","FCFF"],
        "Latest":[Revenue.iloc[-1], NetIncome.iloc[-1], OCF.iloc[-1], FCFF_s.iloc[-1]],
        "TTM":[Revenue_TTM, NetIncome_TTM, OCF_TTM, FCFF_TTM],
        "QoQ":[Rev_QoQ, qoq(NetIncome), qoq(OCF), qoq(FCFF_s)],
        "YoY_TTM":[RevYoY_TTM, yoy_growth_ttm(NetIncome), yoy_growth_ttm(OCF), yoy_growth_ttm(FCFF_s)]
    })
    ttm_growth.to_excel(w, "TTM & Growth", index=False)

    # Quality & Leverage
    quick = (CurrAssets.iloc[-1]/CurrLiab.iloc[-1]) if (not any(pd.isna([CurrAssets.iloc[-1],CurrLiab.iloc[-1]])) and CurrLiab.iloc[-1]!=0) else np.nan
    cash_ratio = (Cash.iloc[-1]/CurrLiab.iloc[-1]) if (not any(pd.isna([Cash.iloc[-1],CurrLiab.iloc[-1]])) and CurrLiab.iloc[-1]!=0) else np.nan
    debt_ratio = (TotLiab.iloc[-1]/TotAssets.iloc[-1]) if (not any(pd.isna([TotLiab.iloc[-1],TotAssets.iloc[-1]])) and TotAssets.iloc[-1]!=0) else np.nan
    roe = (NetIncome_TTM/Equity.iloc[-1]) if (not any(pd.isna([NetIncome_TTM, Equity.iloc[-1]])) and Equity.iloc[-1]!=0) else np.nan
    roa = (NetIncome_TTM/TotAssets.iloc[-1]) if (not any(pd.isna([NetIncome_TTM, TotAssets.iloc[-1]])) and TotAssets.iloc[-1]!=0) else np.nan
    nd = (LT_Debt.iloc[-1]-Cash.iloc[-1]) if not any(pd.isna([LT_Debt.iloc[-1],Cash.iloc[-1]])) else np.nan
    nd_ebitda = (nd/EBITDA_TTM) if (not pd.isna(nd) and EBITDA_TTM and EBITDA_TTM!=0) else np.nan

    quality = pd.DataFrame({
        "Metric":["Quick Ratio","Cash Ratio","Debt Ratio","Net Debt","ND/EBITDA","ROE (TTM)","ROA (TTM)"],
        "Value":[quick, cash_ratio, debt_ratio, nd, nd_ebitda, roe, roa]
    })
    quality.to_excel(w, "Quality & Leverage", index=False)

    # ROIC
    nopat = EBIT.iloc[-1]*(1-tax_rate_eff) if not pd.isna(EBIT.iloc[-1]) else np.nan
    invested_cap = (LT_Debt.iloc[-1] + Equity.iloc[-1] - Cash.iloc[-1]) if not any(pd.isna([LT_Debt.iloc[-1],Equity.iloc[-1],Cash.iloc[-1]])) else np.nan
    roic = (nopat/invested_cap) if (not any(pd.isna([nopat, invested_cap])) and invested_cap!=0) else np.nan
    pd.DataFrame({"Component":["EBIT (latest)","Tax rate","NOPAT","Invested Capital","ROIC"],
                  "Value":[EBIT.iloc[-1],tax_rate_eff,nopat,invested_cap,roic]}).to_excel(w, "ROIC", index=False)

    # FCF
    pd.DataFrame({"Component":["DA (latest)","CapEx (latest, +)","ΔNWC (latest)","FCFF (latest q)","FCFF TTM"],
                  "Value":[DA.iloc[-1], CapEx.iloc[-1], dNWC_latest, FCFF_q, FCFF_TTM]}).to_excel(w, "FCF", index=False)

    # Assumptions / Model / Valuation
    assumptions.to_excel(w, "Assumptions_Model", index=False)
    DCF_tbl.to_excel(w, "ProForma_FCFF_Model", index=False)
    valuation.to_excel(w, "Valuation(Model)", index=False)
    
        # ---------- 전체 시계열 TTM 및 성장률 기록 시트 추가 ----------
    history_tbl = []
    
    for i in range(len(dates)):
        row = {"Date": dates[i]}
    
        def safe_val(series): return series.iloc[i] if i < len(series) else np.nan
    
        row["Revenue"]   = safe_val(Revenue)
        row["NetIncome"] = safe_val(NetIncome)
        row["OCF"]       = safe_val(OCF)
        row["FCFF"]      = safe_val(FCFF_s)
        row["EPS"]       = safe_val(EPS)
    
        # TTM 계산
        if i >= 3:
            row["Revenue_TTM"]   = Revenue.iloc[i-3:i+1].sum()
            row["NetIncome_TTM"] = NetIncome.iloc[i-3:i+1].sum()
            row["OCF_TTM"]       = OCF.iloc[i-3:i+1].sum()
            row["FCFF_TTM"]      = FCFF_s.iloc[i-3:i+1].sum()
            row["EPS_TTM"]       = EPS.iloc[i-3:i+1].sum()
    
        # QoQ
        if i >= 1:
            cur = Revenue.iloc[i]
            prev = Revenue.iloc[i-1]
            row["Revenue_QoQ"] = (cur / prev - 1) if prev != 0 and not pd.isna(prev) else np.nan
    
            cur_eps = EPS.iloc[i]
            prev_eps = EPS.iloc[i-1]
            row["EPS_QoQ"] = (cur_eps / prev_eps - 1) if prev_eps != 0 and not pd.isna(prev_eps) else np.nan
    
        # YoY TTM
        if i >= 7:
            cur = Revenue.iloc[i-3:i+1].sum()
            prev = Revenue.iloc[i-7:i-3].sum()
            row["Revenue_YoY_TTM"] = (cur / prev - 1) if prev != 0 and not pd.isna(prev) else np.nan
    
            cur_ni = NetIncome.iloc[i-3:i+1].sum()
            prev_ni = NetIncome.iloc[i-7:i-3].sum()
            row["NetIncome_YoY_TTM"] = (cur_ni / prev_ni - 1) if prev_ni != 0 and not pd.isna(prev_ni) else np.nan
    
            cur_fcf = FCFF_s.iloc[i-3:i+1].sum()
            prev_fcf = FCFF_s.iloc[i-7:i-3].sum()
            row["FCFF_YoY_TTM"] = (cur_fcf / prev_fcf - 1) if prev_fcf != 0 and not pd.isna(prev_fcf) else np.nan
    
        history_tbl.append(row)
    
    history_df = pd.DataFrame(history_tbl)
    history_df.to_excel(w, "FullHistory_Summary", index=False)


print(f"✅ Saved: {OUTPUT_XLSX}")


In [ ]:
# Historical summary Analyzation

In [ ]:
# -*- coding: utf-8 -*-
"""
{TICKER}_financials_Q.xlsx -> {TICKER}_analysis_Q.xlsx
- 시트명 자동 인식 (Income Statement, Balance Sheet, Cash Flow, Key Financial Ratios)
- CCC, 멀티플, 비율기반 FCFF 예측/DCF
- FullHistory_Summary: 분기별 주요 지표, 재무 비율, 성장률, 주가 병합 포함
"""

import os, re, math
import numpy as np
import pandas as pd

# ========================= 사용자 설정 =========================
TICKER   = "AAPL"
BASE_DIR = r"C:\Users\seung\OneDrive\주식\Financial_Data_real"
INPUT_XLSX  = os.path.join(BASE_DIR, f"{TICKER}_financials_Q.xlsx")
OUTPUT_XLSX = os.path.join(BASE_DIR, f"{TICKER}_analysis_Q.xlsx")
PRICE_CSV_DIR = os.path.join(r"C:\Users\seung\OneDrive\주식\Back Test", TICKER)

CONFIG = {
    "price": None,
    "beta": 1.2,
    "rf": 0.045,
    "mrp": 0.055,
    "pretax_cost_of_debt": 0.05,
    "rev_g_list": None,
    "ebit_margin": None,
    "da_pct": None,
    "capex_pct": None,
    "dnwc_pct": None,
    "tax_rate_override": None,
    "wacc_override": None,
    "ltg": 0.025,
    "exit_multiple": None,
    # 회계연도 종료월 (YoY/PEG 연 1회 계산용) - 필요시 회사에 맞게 변경
    "fye_month": 9
}
# ===============================================================

# --- 유틸 함수 정의
def _norm(s: str) -> str:
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

def pick_sheet(xlsx_path: str, wanted_key: str) -> str:
    aliases = {
        "income_statement":      ["incomestatement", "income", "incomestmt"],
        "balance_sheet":         ["balancesheet", "balance"],
        "cash_flow":             ["cashflowstatement", "cashflow", "cashflowstmt"],
        "key_financial_ratios":  ["keyfinancialratios", "financialratios", "ratios"],
    }
    xl = pd.ExcelFile(xlsx_path)
    norm_map = {name: _norm(name) for name in xl.sheet_names}
    for real_name, normed in norm_map.items():
        for cand in aliases[wanted_key]:
            if cand in normed:
                return real_name
    raise ValueError(f"[{wanted_key}] 시트를 찾지 못했습니다. 시트 목록: {xl.sheet_names}")

def to_num(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return np.nan
    s = str(x).strip()
    if s in ("", "-", "—"): return np.nan
    neg = False
    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1]; neg = True
    s = s.replace(",", "").replace("$", "")
    try:
        v = float(s); return -v if neg else v
    except:
        return np.nan

def norm_cols(df):
    cols = []
    for c in df.columns:
        try:
            d = pd.to_datetime(c, errors="raise")
            cols.append((c, d))
        except:
            pass
    if not cols: return df.iloc[:, 0:0]
    cols.sort(key=lambda t: t[1])
    return df[[c for c,_ in cols]]

def read_sheet(xlsx, wanted_key):
    sheet_name = pick_sheet(xlsx, wanted_key)
    df = pd.read_excel(xlsx, sheet_name=sheet_name)
    if "Metric" not in df.columns:
        df.rename(columns={df.columns[0]:"Metric"}, inplace=True)
    df["Metric"] = df["Metric"].astype(str).str.strip()
    df.set_index("Metric", inplace=True)
    for c in df.columns:
        df[c] = df[c].map(to_num)
    return norm_cols(df)

def get_row(df, keys, default=np.nan):
    if df is None or df.empty:
        return pd.Series([default]*0)
    idx_map = {i.lower().strip(): i for i in df.index}
    low_index = list(idx_map.keys())
    for k in keys:
        lk = k.lower().strip()
        if lk in idx_map: return df.loc[idx_map[lk]]
    for k in keys:
        lk = k.lower().strip()
        for li in low_index:
            if lk in li:
                return df.loc[idx_map[li]]
    return pd.Series([default]*df.shape[1], index=df.columns, name=keys[0])

def ttm(series):
    s = series.dropna()
    return s.iloc[-4:].sum() if len(s)>=4 else np.nan

def yoy_growth_ttm(series):
    s = series.dropna()
    if len(s) < 8: return np.nan
    cur = s.iloc[-4:].sum()
    prv = s.iloc[-8:-4].sum()
    return cur/prv - 1 if prv not in (0, np.nan) else np.nan

def qoq(series):
    s = series.dropna()
    if len(s) < 2: return np.nan
    prev, cur = s.iloc[-2], s.iloc[-1]
    return (cur/prev - 1) if prev not in (0, np.nan) else np.nan

def compute_wacc(price, shares, debt, cash, beta, rf, mrp, cod, tax_rate):
    coe = rf + beta*mrp if (beta is not None and rf is not None and mrp is not None) else None
    mc = price*shares if price and shares else None
    if mc is None or debt is None or (mc + debt)==0 or coe is None:
        return coe, cod, None, mc
    V = mc + debt
    w_e, w_d = mc/V, debt/V
    wacc = w_e*coe + w_d*cod*(1 - (tax_rate if tax_rate is not None else 0.21))
    return coe, cod, wacc, mc

# ---------- 입력 데이터 로드 ----------
inc = read_sheet(INPUT_XLSX, "income_statement")
bal = read_sheet(INPUT_XLSX, "balance_sheet")
cfs = read_sheet(INPUT_XLSX, "cash_flow")
rat = read_sheet(INPUT_XLSX, "key_financial_ratios")

dates = inc.columns
latest = dates[-1] if len(dates)>0 else None

# ---------- 핵심 시리즈 정의 ----------
Revenue   = get_row(inc, ["Revenue"])
COGS      = get_row(inc, ["Cost Of Goods Sold"])
Gross     = get_row(inc, ["Gross Profit"])
EBIT      = get_row(inc, ["EBIT"])
EBITDA    = get_row(inc, ["EBITDA"])
NetIncome = get_row(inc, ["Net Income"])
EPS       = get_row(inc, ["Basic EPS","EPS - Earnings Per Share"])
Shares    = get_row(inc, ["Shares Outstanding","Basic Shares Outstanding"])

OCF       = get_row(cfs, ["Cash Flow From Operating Activities"])
DA        = get_row(cfs, ["Total Depreciation And Amortization"])
CapEx     = -get_row(cfs, ["Net Change In Property, Plant, And Equipment"]).clip(upper=0)

Cash      = get_row(bal, ["Cash On Hand","Cash"])
CurrAssets = get_row(bal, ["Total Current Assets"])
CurrLiab   = get_row(bal, ["Total Current Liabilities"])
LT_Debt    = get_row(bal, ["Long Term Debt","Long-Term Debt"])
TotAssets  = get_row(bal, ["Total Assets"])
TotLiab    = get_row(bal, ["Total Liabilities"])
Equity     = get_row(bal, ["Share Holder Equity","Total Shareholder Equity"])

# ---------- TTM 계산 ----------
Revenue_TTM   = ttm(Revenue)
EBITDA_TTM    = ttm(EBITDA)
EBIT_TTM      = ttm(EBIT)
NetIncome_TTM = ttm(NetIncome)
EPS_TTM       = ttm(EPS)
OCF_TTM       = ttm(OCF)
DA_TTM        = ttm(DA)
CapEx_TTM     = ttm(CapEx)

RevYoY_TTM = yoy_growth_ttm(Revenue)
EPS_QoQ    = qoq(EPS)
Rev_QoQ    = qoq(Revenue)

# ---------- 유효세율 ----------
Tax     = get_row(inc, ["Income Taxes"])
PreTax  = get_row(inc, ["Pre-Tax Income"])
tax_rate_eff = CONFIG["tax_rate_override"]
if tax_rate_eff is None:
    tax_rate_eff = (Tax.iloc[-1]/PreTax.iloc[-1]) if (not pd.isna(Tax.iloc[-1]) and not pd.isna(PreTax.iloc[-1]) and PreTax.iloc[-1]!=0) else 0.21

# ---------- NWC & FCFF ----------
NWC = (CurrAssets - Cash) - CurrLiab
dNWC_latest = NWC.diff().iloc[-1] if len(NWC.dropna())>=2 else np.nan

def fcff_series(EBIT_s, DA_s, CapEx_s, NWC_s, tax_rate):
    out = []
    for i in range(len(EBIT_s)):
        e = EBIT_s.iloc[i]
        da = DA_s.iloc[i] if i < len(DA_s) else np.nan
        cx = CapEx_s.iloc[i] if i < len(CapEx_s) else np.nan
        d_nwc = NWC_s.iloc[i] - NWC_s.iloc[i-1] if i >= 1 and i < len(NWC_s) else 0
        if any(pd.isna([e, da, cx])):
            out.append(np.nan); continue
        out.append(e*(1-tax_rate) + da - cx - d_nwc)
    return pd.Series(out, index=EBIT_s.index)

FCFF_s = fcff_series(EBIT, DA, CapEx, NWC, tax_rate_eff)
FCFF_TTM = ttm(FCFF_s)
FCFF_q = FCFF_s.iloc[-1]

# ---------- CCC (옵션: ratios에 없을 수도 있어 안전 처리) ----------
InvTurn       = get_row(rat, ["Inventory Turnover Ratio"])
RecvTurn      = get_row(rat, ["Receiveable Turnover","Receivable Turnover"])
PayTurn_ratio = get_row(rat, ["Payables Turnover","Accounts Payables Turnover"])
DSO_rat       = get_row(rat, ["Days Sales In Receivables"])
Payables      = get_row(bal, ["Accounts Payable","Trade Payables","Payables"])

COGS_TTM = ttm(COGS)
DIO = (365/InvTurn.iloc[-1]) if (len(InvTurn)>0 and not pd.isna(InvTurn.iloc[-1]) and InvTurn.iloc[-1]!=0) else np.nan
DSO = (DSO_rat.iloc[-1] if (len(DSO_rat)>0 and not pd.isna(DSO_rat.iloc[-1]))
       else (365/RecvTurn.iloc[-1] if (len(RecvTurn)>0 and not pd.isna(RecvTurn.iloc[-1]) and RecvTurn.iloc[-1]!=0) else np.nan))
DPO = np.nan
if len(PayTurn_ratio)>0 and not pd.isna(PayTurn_ratio.iloc[-1]) and PayTurn_ratio.iloc[-1]!=0:
    DPO = 365/PayTurn_ratio.iloc[-1]
elif Payables.notna().any() and COGS_TTM and COGS_TTM!=0:
    ap_series = Payables.dropna()
    avg_ap = ap_series.iloc[-4:].mean() if len(ap_series)>=4 else (ap_series.iloc[-1] if len(ap_series)>0 else np.nan)
    if pd.notna(avg_ap):
        DPO = (avg_ap/COGS_TTM)*365
CCC = DIO + DSO - DPO if all(pd.notna([DIO, DSO, DPO])) else np.nan

# ========================= 엑셀 쓰기 =========================
with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as w:
    # (중략) 다른 시트 저장 로직이 있다면 여기에 추가하세요.

    # ---------- FullHistory_Summary (YoY는 fye_month에 1회) ----------
    dates_dt = pd.to_datetime(dates, errors="coerce")
    fye_month = CONFIG.get("fye_month", 12)

    history_tbl = []
    for i in range(len(dates)):
        dt_i = dates_dt[i]
        row = {"Date": dt_i}
        def sv(s): return s.iloc[i] if i < len(s) else np.nan

        # 원시 분기 값
        row.update({
            "Revenue": sv(Revenue),
            "NetIncome": sv(NetIncome),
            "OCF": sv(OCF),
            "FCFF": sv(FCFF_s),
            "EPS": sv(EPS),
            "Equity": sv(Equity),
            "Assets": sv(TotAssets),
            "Liabilities": sv(TotLiab),
            "Cash": sv(Cash),
            "CurrAssets": sv(CurrAssets),
            "CurrLiab": sv(CurrLiab),
            "Gross": sv(Gross),
            "COGS": sv(COGS),
            "Shares": sv(Shares),
        })

        # TTM & 비율
        if i >= 3:
            rev_ttm  = Revenue.iloc[i-3:i+1].sum()
            ni_ttm   = NetIncome.iloc[i-3:i+1].sum()
            ocf_ttm  = OCF.iloc[i-3:i+1].sum()
            fcf_ttm  = FCFF_s.iloc[i-3:i+1].sum()
            eps_ttm  = EPS.iloc[i-3:i+1].sum()
            gross_ttm = Gross.iloc[i-3:i+1].sum()

            row.update({
                "Revenue_TTM": rev_ttm,
                "NetIncome_TTM": ni_ttm,
                "OCF_TTM": ocf_ttm,
                "FCFF_TTM": fcf_ttm,
                "EPS_TTM": eps_ttm,
                "GrossMargin": (gross_ttm / rev_ttm) if rev_ttm else np.nan,
                "NetMargin": (ni_ttm / rev_ttm) if rev_ttm else np.nan,
                "ROE": (ni_ttm / row["Equity"]) if row["Equity"] not in (0, np.nan) else np.nan,
                "ROA": (ni_ttm / row["Assets"]) if row["Assets"] not in (0, np.nan) else np.nan,
                "Debt_Ratio": (row["Liabilities"] / row["Assets"]) if row["Assets"] not in (0, np.nan) else np.nan,
                "Current_Ratio": (row["CurrAssets"] / row["CurrLiab"]) if row["CurrLiab"] not in (0, np.nan) else np.nan,
                "OCF_to_NetIncome": (ocf_ttm / ni_ttm) if ni_ttm not in (0, np.nan) else np.nan,
                "Market_to_Book": ((CONFIG["price"] * row["Shares"]) / row["Equity"]) if (CONFIG["price"] and row["Shares"] not in (0, np.nan) and row["Equity"] not in (0, np.nan)) else np.nan,
            })

        # QoQ
        if i >= 1:
            row["Revenue_QoQ"] = (Revenue.iloc[i] / Revenue.iloc[i-1] - 1) if Revenue.iloc[i-1] not in (0, np.nan) else np.nan
            row["EPS_QoQ"]     = (EPS.iloc[i] / EPS.iloc[i-1] - 1) if EPS.iloc[i-1] not in (0, np.nan) else np.nan

        # YoY: fye_month에서만
        if (i >= 7) and pd.notna(dt_i) and (dt_i.month == fye_month):
            rev_now = Revenue.iloc[i-3:i+1].sum()
            rev_prev = Revenue.iloc[i-7:i-3].sum()
            eps_now = EPS.iloc[i-3:i+1].sum()
            eps_prev = EPS.iloc[i-7:i-3].sum()
            ni_now = NetIncome.iloc[i-3:i+1].sum()
            ni_prev = NetIncome.iloc[i-7:i-3].sum()
            gross_ttm_now = Gross.iloc[i-3:i+1].sum()

            net_margin_now  = (ni_now / rev_now) if rev_now else np.nan
            net_margin_prev = (ni_prev / rev_prev) if rev_prev else np.nan
            eps_yoy = (eps_now / eps_prev - 1) if eps_prev not in (0, np.nan) else np.nan
            rev_yoy = (rev_now / rev_prev - 1) if rev_prev not in (0, np.nan) else np.nan

            row.update({
                "Revenue_YoY": rev_yoy,
                "EPS_YoY": eps_yoy,
                "NetMargin_YoY": (net_margin_now / net_margin_prev - 1) if net_margin_prev not in (0, np.nan) else np.nan,
                "Revenue_EPS_GrowthRatio": (rev_yoy / eps_yoy) if eps_yoy not in (0, np.nan) else np.nan,
                "GrossMargin_NetMargin_Ratio": ((gross_ttm_now / rev_now) / net_margin_now) if (rev_now and net_margin_now not in (0, np.nan)) else np.nan,
                # PEG는 여기서 계산하지 않음 (가격 CSV 병합 후 계산)
            })
        history_tbl.append(row)

    history_df = pd.DataFrame(history_tbl)

    # ---------- 주가 CSV 병합 ----------
    import glob
    csv_files = glob.glob(os.path.join(PRICE_CSV_DIR, "*.csv"))
    price_path = csv_files[0] if csv_files else None

    if price_path:
        price_df = pd.read_csv(price_path)
        price_df.columns = price_df.columns.str.strip()

        price_col = None
        for c in ["Close", "Adj Close", "종가", "Adj_Close", "Close*", "Price"]:
            if c in price_df.columns:
                price_col = c
                break
        if price_col is None:
            raise ValueError("가격 컬럼을 찾을 수 없습니다. CSV에 Close/Adj Close/종가 등 열이 있는지 확인하세요.")

        price_df = price_df.rename(columns={price_col: "Price"})
        price_df["Date"] = pd.to_datetime(price_df["Date"], errors="coerce")
        price_df = price_df[["Date", "Price"]].dropna()

        history_df["Date"] = pd.to_datetime(history_df["Date"], errors="coerce")
        history_df = history_df.dropna(subset=["Date"])

        merged = pd.merge_asof(
            history_df.sort_values("Date"),
            price_df.sort_values("Date"),
            on="Date",
            direction="backward"
        )
    else:
        merged = history_df.copy()

    # ---------- PEG 계산 (병합 후, fye_month에서만) ----------
    merged["FYE"] = merged["Date"].dt.month.eq(fye_month)

    # P/E(TTM) 먼저 계산(옵션)
    merged["PE_TTM"] = np.where(
        merged.get("EPS_TTM").notna() & (merged.get("EPS_TTM") != 0) & merged.get("Price").notna(),
        merged["Price"] / merged["EPS_TTM"],
        np.nan
    )

    mask_peg = (
        merged["FYE"]
        & merged.get("Price").notna()
        & merged.get("EPS_TTM").notna() & (merged.get("EPS_TTM") != 0)
        & merged.get("EPS_YoY").notna() & (merged.get("EPS_YoY") != 0)
    )

    merged["PEG"] = np.nan
    merged.loc[mask_peg, "PEG"] = (merged.loc[mask_peg, "Price"] / merged.loc[mask_peg, "EPS_TTM"]) / merged.loc[mask_peg, "EPS_YoY"]

    # 저장
    merged.to_excel(w, "FullHistory_Summary", index=False)

print(f"✅ Saved: {OUTPUT_XLSX}")

# This is not financial advice, only data analysis. Please consult a qualified financial professional for personalized guidance.


In [ ]:
#Visualization

In [ ]:
# # ================== Visualization for FullHistory_Summary ==================
# import os
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns

# # 보기 좋은 스타일
# sns.set(style="whitegrid", context="talk")
# plt.rcParams["font.family"] = "DejaVu Sans"  # 한글 폰트가 없다면 영문 대체
# plt.rcParams["figure.dpi"] = 110

# def visualize_full_history(
#     xlsx_path,
#     sheet_name="FullHistory_Summary",
#     outdir="charts",
#     fye_month=9,                   # 🔹 회계연도 종료월: 애플은 9월
#     show_plots=True
# ):
#     # ---------- 데이터 로드 ----------
#     df = pd.read_excel(xlsx_path, sheet_name=sheet_name)
#     df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
#     df = df.sort_values("Date").dropna(subset=["Date"]).reset_index(drop=True)

#     def col(c): return c if c in df.columns else None

#     # 연말(회계연도 종료월)만 True
#     df["FYE"] = df["Date"].dt.month.eq(fye_month)

#     os.makedirs(outdir, exist_ok=True)

#     def save_show(fig, fname):
#         fpath = os.path.join(outdir, fname)
#         fig.tight_layout()
#         fig.savefig(fpath, bbox_inches="tight")
#         if show_plots:
#             plt.show()
#         else:
#             plt.close(fig)
#         print(f"saved: {fpath}")

#     # ---------- 1) 가격 & EPS TTM ----------
#     fig, ax = plt.subplots(figsize=(12, 5))
#     if col("Price"):
#         ax.plot(df["Date"], df["Price"], color="#1f77b4", label="Price")
#         ax.set_ylabel("Price")
#     ax2 = ax.twinx()
#     if col("EPS_TTM"):
#         ax2.plot(df["Date"], df["EPS_TTM"], color="#ff7f0e", label="EPS TTM")
#         ax2.set_ylabel("EPS TTM")
#     ax.set_title("Price vs EPS TTM")
#     lines, labels = ax.get_legend_handles_labels()
#     lines2, labels2 = ax2.get_legend_handles_labels()
#     ax.legend(lines+lines2, labels+labels2, loc="upper left")
#     save_show(fig, "01_price_eps.png")

#     # ---------- 2) 매출/순이익 TTM ----------
#     fig, ax = plt.subplots(figsize=(12, 5))
#     if col("Revenue_TTM"):
#         ax.plot(df["Date"], df["Revenue_TTM"], label="Revenue TTM", color="#2ca02c")
#     if col("NetIncome_TTM"):
#         ax.plot(df["Date"], df["NetIncome_TTM"], label="Net Income TTM", color="#d62728")
#     ax.set_title("Revenue & Net Income (TTM)")
#     ax.set_ylabel("Amount")
#     ax.legend()
#     save_show(fig, "02_revenue_netincome_ttm.png")

#     # ---------- 3) 마진(Gross/Net) ----------
#     fig, ax = plt.subplots(figsize=(12, 5))
#     if col("GrossMargin"):
#         ax.plot(df["Date"], df["GrossMargin"]*100, label="Gross Margin (%)", color="#9467bd")
#     if col("NetMargin"):
#         ax.plot(df["Date"], df["NetMargin"]*100, label="Net Margin (%)", color="#8c564b")
#     ax.set_title("Margins (%)")
#     ax.set_ylabel("%")
#     ax.legend()
#     save_show(fig, "03_margins.png")

#     # ---------- 4) ROE/ROA & Debt Ratio ----------
#     fig, ax = plt.subplots(figsize=(12, 5))
#     if col("ROE"):
#         ax.plot(df["Date"], df["ROE"]*100, label="ROE (%)", color="#17becf")
#     if col("ROA"):
#         ax.plot(df["Date"], df["ROA"]*100, label="ROA (%)", color="#bcbd22")
#     ax.set_ylabel("%")
#     ax2 = ax.twinx()
#     if col("Debt_Ratio"):
#         ax2.plot(df["Date"], df["Debt_Ratio"]*100, label="Debt Ratio (%)", color="#7f7f7f", linestyle="--")
#         ax2.set_ylabel("Debt Ratio (%)")
#     ax.set_title("ROE / ROA & Debt Ratio")
#     lines, labels = ax.get_legend_handles_labels()
#     lines2, labels2 = ax2.get_legend_handles_labels()
#     ax.legend(lines+lines2, labels+labels2, loc="upper left")
#     save_show(fig, "04_roe_roa_debt.png")

#     # ---------- 5) OCF/FCFF TTM & OCF/NetIncome ----------
#     fig, ax = plt.subplots(figsize=(12, 5))
#     if col("OCF_TTM"):
#         ax.plot(df["Date"], df["OCF_TTM"], label="OCF TTM", color="#1f77b4")
#     if col("FCFF_TTM"):
#         ax.plot(df["Date"], df["FCFF_TTM"], label="FCFF TTM", color="#ff7f0e")
#     ax.set_ylabel("Amount")
#     ax2 = ax.twinx()
#     if col("OCF_to_NetIncome"):
#         ax2.plot(df["Date"], df["OCF_to_NetIncome"], label="OCF / NetIncome (TTM)", color="#2ca02c", linestyle="--")
#         ax2.set_ylabel("OCF / NetIncome (x)")
#     ax.set_title("Cash Flows & Quality")
#     lines, labels = ax.get_legend_handles_labels()
#     lines2, labels2 = ax2.get_legend_handles_labels()
#     ax.legend(lines+lines2, labels+labels2, loc="upper left")
#     save_show(fig, "05_cashflows_quality.png")

#     # ---------- 6) 성장률: QoQ + YoY(연말 마커) ----------
#     fig, ax = plt.subplots(figsize=(12, 5))
#     if col("Revenue_QoQ"):
#         ax.bar(df["Date"], df["Revenue_QoQ"]*100, width=60, label="Revenue QoQ (%)", color="#6baed6")
#     if col("EPS_QoQ"):
#         ax.bar(df["Date"], df["EPS_QoQ"]*100, width=30, label="EPS QoQ (%)", color="#9ecae1")
#     if col("Revenue_YoY"):
#         yoy_mask = df["FYE"] & df["Revenue_YoY"].notna()
#         ax.scatter(df.loc[yoy_mask, "Date"], df.loc[yoy_mask, "Revenue_YoY"]*100,
#                    color="#d62728", label="Revenue YoY (FYE)", zorder=5)
#     if col("EPS_YoY"):
#         yoy_mask2 = df["FYE"] & df["EPS_YoY"].notna()
#         ax.scatter(df.loc[yoy_mask2, "Date"], df.loc[yoy_mask2, "EPS_YoY"]*100,
#                    color="#2ca02c", label="EPS YoY (FYE)", zorder=5, marker="^")
#     ax.axhline(0, color="black", linewidth=0.8)
#     ax.set_ylabel("%")
#     ax.set_title("Growth: QoQ & YoY (annual)")
#     ax.legend()
#     save_show(fig, "06_growth_qoq_yoy.png")

#     # ---------- 7) PEG & Price ----------
#     fig, ax = plt.subplots(figsize=(12, 5))
#     if col("PEG"):
#         ax.plot(df["Date"], df["PEG"], label="PEG", color="#ff9896")
#         peg_mask = df["FYE"] & df["PEG"].notna()
#         ax.scatter(df.loc[peg_mask, "Date"], df.loc[peg_mask, "PEG"], 
#                    color="#c70039", s=25, zorder=5, label="PEG (FYE)")
#         ax.set_ylabel("PEG")
#     ax2 = ax.twinx()
#     if col("Price"):
#         ax2.plot(df["Date"], df["Price"], label="Price", color="#1f77b4", alpha=0.7)
#         ax2.set_ylabel("Price")
#     ax.set_title("PEG & Price")
#     lines, labels = ax.get_legend_handles_labels()
#     lines2, labels2 = ax2.get_legend_handles_labels()
#     ax.legend(lines+lines2, labels+labels2, loc="upper left")
#     save_show(fig, "07_peg_price.png")

#     # ---------- 8) 유동성 & 레버리지 ----------
#     fig, ax = plt.subplots(figsize=(12, 5))
#     if col("Current_Ratio"):
#         ax.plot(df["Date"], df["Current_Ratio"], label="Current Ratio (x)", color="#2ca02c")
#         ax.axhline(1.0, color="#2ca02c", linestyle="--", linewidth=0.9)
#     if col("Debt_Ratio"):
#         ax2 = ax.twinx()
#         ax2.plot(df["Date"], df["Debt_Ratio"]*100, label="Debt Ratio (%)", color="#7f7f7f")
#         ax2.set_ylabel("Debt Ratio (%)")
#     ax.set_title("Liquidity & Leverage")
#     ax.set_ylabel("Current Ratio (x)")
#     lines, labels = ax.get_legend_handles_labels()
#     if "ax2" in locals():
#         lines2, labels2 = ax2.get_legend_handles_labels()
#         ax.legend(lines+lines2, labels+labels2, loc="upper left")
#     else:
#         ax.legend()
#     save_show(fig, "08_liquidity_leverage.png")

#     # ---------- 9) Revenue vs EPS Growth Ratio ----------
#     if col("Revenue_EPS_GrowthRatio"):
#         fig, ax = plt.subplots(figsize=(12, 4.5))
#         mask = df["FYE"] & df["Revenue_EPS_GrowthRatio"].notna()
#         ax.plot(df.loc[mask, "Date"], df.loc[mask, "Revenue_EPS_GrowthRatio"], 
#                 marker="o", color="#9467bd")
#         ax.axhline(1.0, color="#444", linestyle="--", linewidth=0.9)
#         ax.set_title("Revenue Growth / EPS Growth (FYE)")
#         ax.set_ylabel("Ratio (x)")
#         save_show(fig, "09_rev_eps_growth_ratio.png")

#     print("✅ All charts generated.")

# # ================== 실행 예시 ==================
# xlsx_path = r"C:\Users\seung\OneDrive\주식\Financial_Data_real\AAPL_analysis_Q.xlsx"
# visualize_full_history(
#     xlsx_path,
#     sheet_name="FullHistory_Summary",
#     outdir=r"C:\Users\seung\OneDrive\주식\Financial_Data_real\charts",
#     fye_month=9,   # 🔹 애플은 9월 결산
#     show_plots=True
# )


In [ ]:
# ================== Visualization for FullHistory_Summary ==================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 보기 좋은 스타일
sns.set(style="whitegrid", context="talk")
plt.rcParams["font.family"] = "DejaVu Sans"  # 한글 폰트가 없다면 영문 대체
plt.rcParams["figure.dpi"] = 110

def visualize_full_history(
    xlsx_path,
    sheet_name="FullHistory_Summary",
    outdir="charts",
    fye_month=9,                   # 🔹 회계연도 종료월: 애플은 9월
    show_plots=True
):
    # ---------- 데이터 로드 ----------
    df = pd.read_excel(xlsx_path, sheet_name=sheet_name)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.sort_values("Date").dropna(subset=["Date"]).reset_index(drop=True)

    def has(c): return c in df.columns

    # 연말(회계연도 종료월)만 True
    df["FYE"] = df["Date"].dt.month.eq(fye_month)

    # PEG Growth (FYE 행에서만 PEG의 전년 대비 %변화)
    if has("PEG"):
        peg_fye = df.loc[df["FYE"], ["Date", "PEG"]].copy()
        peg_fye["PEG_Growth"] = peg_fye["PEG"].pct_change()
        df["PEG_Growth"] = np.nan
        df.loc[df["FYE"], "PEG_Growth"] = peg_fye["PEG_Growth"].values

    os.makedirs(outdir, exist_ok=True)

    def save_show(fig, fname):
        fpath = os.path.join(outdir, fname)
        fig.tight_layout()
        fig.savefig(fpath, bbox_inches="tight")
        if show_plots:
            plt.show()
        else:
            plt.close(fig)
        print(f"saved: {fpath}")

    # ---------- 1) 가격 & EPS TTM ----------
    fig, ax = plt.subplots(figsize=(12, 5))
    if has("Price"):
        ax.plot(df["Date"], df["Price"], color="#1f77b4", label="Price")
        ax.set_ylabel("Price")
    ax2 = ax.twinx()
    if has("EPS_TTM"):
        ax2.plot(df["Date"], df["EPS_TTM"], color="#ff7f0e", label="EPS TTM")
        ax2.set_ylabel("EPS TTM")
    ax.set_title("Price vs EPS TTM")
    l1, lab1 = ax.get_legend_handles_labels()
    l2, lab2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, lab1+lab2, loc="upper left")
    save_show(fig, "01_price_eps.png")

    # ---------- 2) 매출/순이익 TTM ----------
    fig, ax = plt.subplots(figsize=(12, 5))
    if has("Revenue_TTM"):
        ax.plot(df["Date"], df["Revenue_TTM"], label="Revenue TTM", color="#2ca02c")
    if has("NetIncome_TTM"):
        ax.plot(df["Date"], df["NetIncome_TTM"], label="Net Income TTM", color="#d62728")
    ax.set_title("Revenue & Net Income (TTM)")
    ax.set_ylabel("Amount")
    ax.legend()
    save_show(fig, "02_revenue_netincome_ttm.png")

    # ---------- 3) 마진(Gross/Net) ----------
    fig, ax = plt.subplots(figsize=(12, 5))
    if has("GrossMargin"):
        ax.plot(df["Date"], df["GrossMargin"]*100, label="Gross Margin (%)", color="#9467bd")
    if has("NetMargin"):
        ax.plot(df["Date"], df["NetMargin"]*100, label="Net Margin (%)", color="#8c564b")
    ax.set_title("Margins (%)")
    ax.set_ylabel("%")
    ax.legend()
    save_show(fig, "03_margins.png")

    # ---------- 4) ROE/ROA & Debt Ratio ----------
    fig, ax = plt.subplots(figsize=(12, 5))
    if has("ROE"):
        ax.plot(df["Date"], df["ROE"]*100, label="ROE (%)", color="#17becf")
    if has("ROA"):
        ax.plot(df["Date"], df["ROA"]*100, label="ROA (%)", color="#bcbd22")
    ax.set_ylabel("%")
    ax2 = ax.twinx()
    if has("Debt_Ratio"):
        ax2.plot(df["Date"], df["Debt_Ratio"]*100, label="Debt Ratio (%)", color="#7f7f7f", linestyle="--")
        ax2.set_ylabel("Debt Ratio (%)")
    ax.set_title("ROE / ROA & Debt Ratio")
    l1, lab1 = ax.get_legend_handles_labels()
    l2, lab2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, lab1+lab2, loc="upper left")
    save_show(fig, "04_roe_roa_debt.png")

    # ---------- 5) Cash Flows & Quality ----------
    fig, ax = plt.subplots(figsize=(12, 5))
    if has("OCF_TTM"):
        ax.plot(df["Date"], df["OCF_TTM"], label="OCF TTM", color="#1f77b4")
    if has("FCFF_TTM"):
        ax.plot(df["Date"], df["FCFF_TTM"], label="FCFF TTM", color="#ff7f0e")
    ax.set_ylabel("Amount")
    ax2 = ax.twinx()
    if has("OCF_to_NetIncome"):
        ax2.plot(df["Date"], df["OCF_to_NetIncome"], label="OCF / NetIncome (TTM)", color="#2ca02c", linestyle="--")
        ax2.set_ylabel("OCF / NetIncome (x)")
    ax.set_title("Cash Flows & Quality")
    l1, lab1 = ax.get_legend_handles_labels()
    l2, lab2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, lab1+lab2, loc="upper left")
    save_show(fig, "05_cashflows_quality.png")

    # ---------- 6) 성장률: QoQ ----------
    fig, ax = plt.subplots(figsize=(12, 5))
    if has("Revenue_QoQ"):
        ax.bar(df["Date"], df["Revenue_QoQ"]*100, width=60, label="Revenue QoQ (%)", color="#6baed6")
    if has("EPS_QoQ"):
        ax.bar(df["Date"], df["EPS_QoQ"]*100, width=30, label="EPS QoQ (%)", color="#9ecae1")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("%")
    ax.set_title("Growth: QoQ (quarter-over-quarter)")
    ax.legend()
    save_show(fig, "06_qoq_only.png")

    # ---------- 7) 성장률: YoY ----------
    fig, ax = plt.subplots(figsize=(12, 5))
    if has("Revenue_YoY"):
        m1 = df["FYE"] & df["Revenue_YoY"].notna()
        ax.stem(df.loc[m1, "Date"], df.loc[m1, "Revenue_YoY"]*100,
                linefmt="#d62728", markerfmt="o", basefmt="k-", label="Revenue YoY (FYE)")
    if has("EPS_YoY"):
        m2 = df["FYE"] & df["EPS_YoY"].notna()
        ax.stem(df.loc[m2, "Date"], df.loc[m2, "EPS_YoY"]*100,
                linefmt="#2ca02c", markerfmt="^", basefmt="k-", label="EPS YoY (FYE)")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("%")
    ax.set_title("Growth: YoY (FYE only)")
    ax.legend()
    save_show(fig, "07_yoy_only.png")

    # ---------- 8) PEG ----------
    fig, ax = plt.subplots(figsize=(12, 5))
    if has("PEG"):
        ax.plot(df["Date"], df["PEG"], color="#ff9896", alpha=0.55, label="PEG (all rows)")
        m = df["FYE"] & df["PEG"].notna()
        ax.scatter(df.loc[m, "Date"], df.loc[m, "PEG"], color="#c70039", s=28, zorder=5, label="PEG (FYE)")
        ax.set_ylabel("PEG")
    ax.set_title("PEG (FYE highlighted)")
    ax.legend()
    save_show(fig, "08_peg_only.png")

    # ---------- 9) PEG Growth ----------
    if has("PEG_Growth"):
        fig, ax = plt.subplots(figsize=(12, 4.5))
        mg = df["FYE"] & df["PEG_Growth"].notna()
        ax.bar(df.loc[mg, "Date"], df.loc[mg, "PEG_Growth"]*100, color="#e377c2", width=120, label="PEG Growth YoY (FYE)")
        ax.axhline(0, color="#444", linestyle="--", linewidth=0.9)
        ax.set_ylabel("%")
        ax.set_title("PEG Growth YoY (FYE only)")
        ax.legend()
        save_show(fig, "09_peg_growth.png")

    # ---------- 10) 유동성 & 레버리지 ----------
    fig, ax = plt.subplots(figsize=(12, 5))
    if has("Current_Ratio"):
        ax.plot(df["Date"], df["Current_Ratio"], label="Current Ratio (x)", color="#2ca02c")
        ax.axhline(1.0, color="#2ca02c", linestyle="--", linewidth=0.9)
    if has("Debt_Ratio"):
        ax2 = ax.twinx()
        ax2.plot(df["Date"], df["Debt_Ratio"]*100, label="Debt Ratio (%)", color="#7f7f7f")
        ax2.set_ylabel("Debt Ratio (%)")
    ax.set_title("Liquidity & Leverage")
    ax.set_ylabel("Current Ratio (x)")
    l1, lab1 = ax.get_legend_handles_labels()
    if "ax2" in locals():
        l2, lab2 = ax2.get_legend_handles_labels()
        ax.legend(l1+l2, lab1+lab2, loc="upper left")
    else:
        ax.legend()
    save_show(fig, "10_liquidity_leverage.png")

    # ---------- 11) Revenue vs EPS Growth Ratio ----------
    if has("Revenue_EPS_GrowthRatio"):
        fig, ax = plt.subplots(figsize=(12, 4.5))
        mask = df["FYE"] & df["Revenue_EPS_GrowthRatio"].notna()
        ax.plot(df.loc[mask, "Date"], df.loc[mask, "Revenue_EPS_GrowthRatio"],
                marker="o", color="#9467bd")
        ax.axhline(1.0, color="#444", linestyle="--", linewidth=0.9)
        ax.set_title("Revenue Growth / EPS Growth (FYE)")
        ax.set_ylabel("Ratio (x)")
        save_show(fig, "11_rev_eps_growth_ratio.png")

    print("✅ All charts generated.")

================== 실행 예시 ==================
xlsx_path = r"C:\Users\seung\OneDrive\주식\Financial_Data_real\AAPL_analysis_Q.xlsx"
visualize_full_history(
    xlsx_path,
    sheet_name="FullHistory_Summary",
    outdir=r"C:\Users\seung\OneDrive\주식\Financial_Data_real\charts",
    fye_month=9,   # 🔹 애플은 9월 결산
    show_plots=True
)
